In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
sample_sub = pd.read_csv("/kaggle/input/playground-series-s6e2/sample_submission.csv")

print(train.shape)
train.head()


(630000, 15)


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [3]:
train.drop("id", axis=1, inplace=True)
test_ids = test["id"]
test.drop("id", axis=1, inplace=True)


In [4]:
train["Heart Disease"].value_counts(normalize=True)


Heart Disease
Absence     0.55166
Presence    0.44834
Name: proportion, dtype: float64

In [5]:
train["Heart Disease"] = train["Heart Disease"].map({
    "Absence": 0,
    "Presence": 1
})


In [6]:
train["Heart Disease"].value_counts()


Heart Disease
0    347546
1    282454
Name: count, dtype: int64

In [7]:
train.dtypes


Age                          int64
Sex                          int64
Chest pain type              int64
BP                           int64
Cholesterol                  int64
FBS over 120                 int64
EKG results                  int64
Max HR                       int64
Exercise angina              int64
ST depression              float64
Slope of ST                  int64
Number of vessels fluro      int64
Thallium                     int64
Heart Disease                int64
dtype: object

In [8]:


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier


# Interaction features
train["age_chol"] = train["Age"] * train["Cholesterol"]
test["age_chol"] = test["Age"] * test["Cholesterol"]

train["hr_age"] = train["Max HR"] / (train["Age"] + 1)
test["hr_age"] = test["Max HR"] / (test["Age"] + 1)

train["bp_age"] = train["BP"] / (train["Age"] + 1)
test["bp_age"] = test["BP"] / (test["Age"] + 1)


# SPLIT FEATURES

X = train.drop("Heart Disease", axis=1)
y = train["Heart Disease"]


# CROSS VALIDATION SETUP

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

test_lgb = np.zeros(len(test))
test_xgb = np.zeros(len(test))


# TRAINING LOOP

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    
    print(f"\n===== Fold {fold+1} =====")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
# LIGHTGBM MODEL
    lgb_model = LGBMClassifier(
        n_estimators=4000,
        learning_rate=0.02,
        num_leaves=256,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=1,
        random_state=42,
        n_jobs=-1
    )
    
    lgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[
            early_stopping(200),
            log_evaluation(200)
        ]
    )
    
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_lgb += lgb_model.predict_proba(test)[:, 1] / 5
    
   #xg boost
    xgb_model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        random_state=42,
        use_label_encoder=False,
        n_jobs=-1
    )
    
    xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb += xgb_model.predict_proba(test)[:, 1] / 5
#cv scores
print("\nLightGBM CV AUC:", roc_auc_score(y, oof_lgb))
print("XGBoost CV AUC:", roc_auc_score(y, oof_xgb))

#blending
final_oof = 0.6 * oof_lgb + 0.4 * oof_xgb
final_test_preds = 0.6 * test_lgb + 0.4 * test_xgb

print("Blended CV AUC:", roc_auc_score(y, final_oof))

#submission
submission = pd.DataFrame({
    "id": test_ids,
    "Heart Disease": final_test_preds
})

submission.to_csv("submission.csv", index=False)

print("\nSubmission file created successfully!")



===== Fold 1 =====
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.046444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1187
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 16
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.954163	valid_0's binary_logloss: 0.274123
[400]	valid_0's auc: 0.954767	valid_0's binary_logloss: 0.269724
[600]	valid_0's auc: 0.954945	valid_0's binary_logloss: 0.26916
[80

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:00:15] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== Fold 2 =====
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034675 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1182
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 16
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.953368	valid_0's binary_logloss: 0.276542
[400]	valid_0's auc: 0.95397	valid_0's binary_logloss: 0.272295
[600]	valid_0's auc: 0.954059	valid_0's binary_logloss: 0.272013
Ear

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:03:47] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== Fold 3 =====
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1179
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 16
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.954081	valid_0's binary_logloss: 0.27462
[400]	valid_0's auc: 0.954754	valid_0's binary_logloss: 0.269853
[600]	valid_0's auc: 0.954903	valid_0's binary_logloss: 0.269306
[80

...


===== Fold 4 =====
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.086303 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1181
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 16
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.953548	valid_0's binary_logloss: 0.276058
[400]	valid_0's auc: 0.954196	valid_0's binary_logloss: 0.271564
[600]	valid_0's auc: 0.954367	valid_0's binary_logloss: 0.271008
Early stopping, best iteration is:
[553]	valid_0's auc: 0.954373	v

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:11:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== Fold 5 =====
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225964, number of negative: 278036
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.101055 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1182
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 16
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448341 -> initscore=-0.207375
[LightGBM] [Info] Start training from score -0.207375
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.95442	valid_0's binary_logloss: 0.273719
[400]	valid_0's auc: 0.95509	valid_0's binary_logloss: 0.268891
[600]	valid_0's auc: 0.955259	valid_0's binary_logloss: 0.268299
Early stopping, best iteration is:
[542]	valid_0's auc: 0.955268	val

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:14:41] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



LightGBM CV AUC: 0.9547133835481284
XGBoost CV AUC: 0.954883934913094
Blended CV AUC: 0.954954377052482

Submission file created successfully!
